# HACK AI / Intro to Large Language Modelling
## BERT for supervised multi-class classification

### *Mariam Cook*

### *m.cook6@exeter.ac.uk*

### *University of Exeter Centre for Computational Social Science*


# Our goals for this afternoon



1. Learn about the architecture behind Large Language Models (LLMs)
2. Load some pre-labelled text examples from UK parliament debates, and split them into a training and test dataset
3. Fine-tune the BERT LLM to recognise whether each debate sample is from a Conservate or Labour party member (at this time the Labour party were in opposition)
4. Review the results from our model training, by comparing its predictions to the true labels in the test set
5. Save our fine-tuned model to Google Drive
6. Load the model into a new notebook and check it is working


THIS IS A HANDS-ON, interactive session, to get the most out of it, run the code as we go along, complete the exercises, and ask questions.



## Transformer / Large Language Models (LLMs): a brief overview

1. Text embeddings are numerical vectors that represent a series of words or word-pieces (usually referred to as tokens). They are calculated via machine learning counting and prediction techniques. LLM foundation models leverage high dimensional embeddings determined via unsupervised neural network training over massive volumes of natural langauge.
2. A key advance in LLM architecture was introduced in the 2017 *Attention is all you need* paper. This proposed 'self-attention' and positional encoding for word embedding creation, which allows long-range dependencies between words, and hence their *relative meaning* to be efficiently captured.
3. Self-attention describes how an embedding for each token is calculated based on itself aswell as tokens surrounding it within a text span (its context window). Thus, words, or word-pieces, in text, are considered in terms of their attendance, or importance, to one another. This is achieved via matrix multiplication of queries (smaller dimension than embedding vector) and keys (answers to queries) so that the embeddings for each token, for each language feature, encompass attendance to other tokens.  
4. The BERT Large Language Model is pre-trained on unsupervised *guess the word (masked language model)* and *next sentence prediction* tasks. Its neural network architecture is comprised of an encoder layer and a feed forward layer.
5. LLMs are sometimes called generative because they can predict the next token in a sequence, given some text, they can 'generate' this.
6. Pre-trained LLMs, designed for sequence to sequence modelling (such as language translation) can be fine-tuned for many downstream NLP tasks such as: *our obective here, class prediction*. This requires some pre-labelled text units, often referred to as text annotations.
7. Pre-trained LLMs perform better than previously common methods using a 'bag of words' that do not take into account language grammar and context, and word2vec, that leverages pre-trained embeddings, but where there is only one (non-contextual) embedding associated with each word e.g. (financial) bank and (river) bank would have the same embedded representation.
8. “A probability distribution over words or word sequences is the fundamental building block of a language model. In application, a language model provides the chance that a particular word sequence can be considered ‘valid’... [i.e.] similar to the way people speak (or, to be more specific, write) because this is how the language model acquires its knowledge.” Shashank Jain (2022)
9. Today's commercial LLMs or AI models incorporate LLMs as part of multi-step pipelines involving human labelled data, reinforcement learning, and reinforcement learning with human feedback (RLHF).

### Let's learn a bit more about transformers, embeddings and attention
We will watch the first 6 and a half minutes, go back and watch the whole series if interested

https://www.youtube.com/watch?v=eMlx5fFNoYc

In [ ]:
# insert video

from IPython.display import YouTubeVideo

YouTubeVideo('eMlx5fFNoYc', width = 800, height = 450)


**Reference materials:**


*   Attention Is All You Need - https://doi.org/10.48550/arXiv.1706.03762

*   The Illustrated Transformer - http://jalammar.github.io/illustrated-transformer/

*   Stanford CS224N: Natural Language Processing with Deep Learning Playlist - https://www.youtube.com/playlist?list=PLoROMvodv4rOSH4v6133s9LFPRHjEmbmJ

* Credit: Workflow developed here originally based on James Briggs and Sentdex tutorials - https://www.youtube.com/watch?v=pjtnkCGElcE /  https://www.youtube.com/@sentdex/videos. Tensorflow to Pytorch framework conversion and statistical review with help of Claude Code.


## Let's get started

To use a large language model, underpinned by computational neural networks, we need to run a GPU. In the top right on colab, 'change runtime type' to TPU, then run the next cell

In [ ]:
# First check the GPU is running on Colab (Graphics Processing Unit, we use due to need for massive parallel processing)
import tensorflow as tf
device_name = tf.test.gpu_device_name()
if device_name != '/device:GPU:0':
  raise SystemError('GPU device not found')
print('Found GPU at: {}'.format(device_name))

Install transformers from the hugging face library

In [ ]:
# https://huggingface.co/docs/transformers/main/installation

!pip install -q transformers
import transformers # we have installed but we also need to import
import pandas as pd # we also need the pandas library for building dataframes and working with our data

## Let's read in a dataset of labelled data

Here is the sources of this data: 20 UK Parliament debates.
To learn how to scrape them go to this notebook: https://github.com/TristanCann/HackAI-Intro-NLP-scraping/blob/main/Scrape_webpages_parliamentary_debates.ipynb




In [ ]:
''' In case you are curious here are all the links to the debates:

mylinks = ["https://www.theyworkforyou.com/whall/?id=2023-06-06b.309.0", # 1 Net Zero: 2050 Target
                    "https://www.theyworkforyou.com/debates/?id=2023-06-14b.312.3", # 2 cost of living committee
                    "https://www.theyworkforyou.com/debates/?id=2023-06-19b.569.0", # 3 stop and search
                    "https://www.theyworkforyou.com/whall/?id=2023-06-21a.384.0", # 4 ultra-processed food
                    "https://www.theyworkforyou.com/lords/?id=2023-06-19a.5.2", # 5 Primary care - inequality
                    "https://www.theyworkforyou.com/lords/?id=2023-06-15b.2101.2", # 6 Ukraine: Ministry of Defence Strategy - Question
                    "https://www.theyworkforyou.com/debates/?id=2023-06-15a.474.1", # 7 Pride Month
                    "https://www.theyworkforyou.com/debates/?id=2023-06-21b.801.3", # 8 Schedule - Minimum service levels for certain strikes
                    "https://www.theyworkforyou.com/whall/?id=2023-06-21a.352.0", # 9 Tackling Loneliness and Connecting Communities — [Dr Rupa Huq in the Chair]
                    "https://www.theyworkforyou.com/debates/?id=2023-06-22a.940.0", # 10 Business of the House – in the House of Commons at 10:30 am on 22 June 2023.
                    "https://www.theyworkforyou.com/debates/?id=2023-06-21b.849.1", # 11 Animal Welfare (Kept Animals) – in the House of Commons at 3:13 pm on 21 June 2023.
                    "https://www.theyworkforyou.com/debates/?id=2023-06-21b.786.6", # 12 Engagements Prime Minister – in the House of Commons on 21 June 2023.
                    "https://www.theyworkforyou.com/whall/?id=2023-06-19a.227.0", # 13 Cost of Living: Parental Leave and Pay — [Steve McCabe in the Chair]
                    "https://www.theyworkforyou.com/debates/?id=2023-06-20c.701.0", # 14 Cost of Living support
                    "https://www.theyworkforyou.com/debates/?id=2023-06-15a.514.0", # 15 Migration – in the House of Commons at 3:17 pm on 15 June 2023.
                    "https://www.theyworkforyou.com/debates/?id=2023-06-15a.445.0", # 16 Business of the House – in the House of Commons at 11:02 am on 15 June 2023
                    "https://www.theyworkforyou.com/pbc/2022-23/Energy_Bill/14-0_2023-06-22a.381.2", # 17 Clause 270 - Prohibition of new coal mines
                    "https://www.theyworkforyou.com/debates/?id=2023-06-14b.365.0", # 18 Global Military Operations 14 June 2023
                    "https://www.theyworkforyou.com/debates/?id=2023-06-14b.280.4", # 19 Regional Innovation
                    "https://www.theyworkforyou.com/whall/?id=2023-06-13a.100.0" # Tackling Rogue Builders in Westminster Hall at 4:00 pm on 13 June 2023.
                    ]

In [ ]:
# this dataset contains text from transcripts of UK Parliament Debates, I recently scraped from theyworkforyou.com using the beautiful soup library. See the notebook linked above for help scraping your own text.

all_data = pd.read_csv('con_lab.csv', index_col=0)
all_data

## Let's look into our dataframe at a particular row

In [ ]:
all_data.loc[1].values

**Exercise 1:** print out the values of a Labour party row from the dataframe

In [ ]:
# all values for a row
all_data.loc[all_data['Party'] == 'Labour'].values[0]

In [ ]:
# get a random row nicely printed
all_data.loc[all_data['Party'] == 'Labour'].sample(1)

**Exercise 2:** What are the unique names of the Conservative party speakers?

In [ ]:
print(len(pd.unique(all_data.Name.values)))
pd.unique(all_data.Name.values)

# We are interested in predicting the Party (label) based on a string of text

In [ ]:
# new dataframe with the Party and debate text only, for ease of viewing
df2 = all_data[['Party', 'Text']]
df2

**Exercise 3:** How many statements do we have from members of each Party (How many Labour, how many Conservative)?

In [ ]:
df2.Party.value_counts()

## Split data into a training (df) and test set (testdf) stratified across classes



In [ ]:
# we are keeping aside 20% for a test set. An alternative approach is cross-fold validation
# Stratify ensures we are splitting items from each label across the training and test set, at the same proportion
# Specify random state 42 to retain consistency for each run / we should see the same split

import numpy as np # for statistical operations
from sklearn.model_selection import train_test_split

df, test_df = train_test_split(df2, test_size=0.20, random_state = 42, stratify=df2["Party"])

In [ ]:
df

In [ ]:
test_df

In [ ]:
(len(df)+len(test_df)) == len(df2)

## For BERT transformer modelling label values must be integers, let's turn labels into these

In [ ]:
def turn_labels_into_integers(a_df_column):
    list_from_column = list(a_df_column)
    all_classes=set(a_df_column) # this gets the unique set of classes
    class_counter=0
    int_labels = []
    class_int_dict={}
    for a_class in all_classes:
        class_int_dict[a_class]=class_counter
        class_counter+=1
    for a_label in list_from_column:
        int_labels.append(class_int_dict[a_label])
    return int_labels, class_int_dict

In [ ]:
new_coded_labels,class_label_dict = turn_labels_into_integers(df['Party'])

In [ ]:
class_label_dict # this is our reference dictionary for each coded class (it does not matter which number is assigned to which class, but we will need to know and use this mapping later)

In [ ]:
df['Party'] = new_coded_labels # update the 'Party' column in our training dataframe to the values of the newly coded integer labels

In [ ]:
df

## Prepare the data for the transformer

Initialize empty zero arrays

In [ ]:
seq_len = 512 # This is the number of characters in a span of text; with BERT we are restricted to 512
num_samples = len(df)

Xids = np.zeros((num_samples, seq_len))
Xmask = np.zeros((num_samples, seq_len))

Xids.shape # print out the shape of the first zero array

In [ ]:
Xids

Exercise 4: Why does our matrix have 512 columns?

Answer: (edit this text cell)

In [ ]:
from transformers import BertTokenizer

# we are using the cased version
tokenizer = BertTokenizer.from_pretrained('bert-base-cased')

In [ ]:
!pip install transformers torch -q

In [ ]:
import torch
import torch.nn as nn
from transformers import BertModel, AutoConfig

In [ ]:
for i, text_item in enumerate(df['Text']):
    tokens = tokenizer(text_item, max_length=seq_len, truncation=True,
                       padding='max_length', add_special_tokens=True,
                       return_tensors='pt')
    Xids[i, :] = tokens['input_ids'][0]
    Xmask[i, :] = tokens['attention_mask'][0]

In [ ]:
Xids # tokenized text - each word/subword has been mapped to a unique integer ID from BERT's vocabulary. 101 is always the start of a sequence. BERT needs to be fed these numerical representations

In [ ]:
# look deeper into the matrix

Xids[0,:] # if the text is shorter than 512 characters the matrix at that position has zero value

In [ ]:
Xmask # attention layer - wherever there is a 1, BERT will calculate attention, so avoids attention to padding tokens

In [ ]:
labels = df['Party'].values  # shape: (num_samples,) - just the raw class indices
labels

In [ ]:
labels.shape

In [ ]:
import torch # we are using the pytorch framework
from torch.utils.data import TensorDataset

dataset = TensorDataset(
    torch.tensor(Xids, dtype=torch.long),
    torch.tensor(Xmask, dtype=torch.long),
    torch.tensor(labels, dtype=torch.long)  # long for class indices
)

In [ ]:
Xmask[0,:].shape

In [ ]:
Xids[0,:].shape

In [ ]:
labels.shape

### Split dataset and define model

In [ ]:
from torch.utils.data import DataLoader, random_split
from transformers import AutoModel
import torch.nn as nn

batch_size = 8

# Split dataset
split = 0.9
train_size = int(num_samples * split)
val_size = num_samples - train_size

train_ds, val_ds = random_split(dataset, [train_size, val_size])

# Create dataloaders (PyTorch's equivalent of batching/shuffling)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, drop_last=True)

# Define model

#Input: [batch, 512 dimensions] → BERT → [batch, 768 dimensions] (Comprising 12 layers ) → Dense → [batch, 1024 dimensions] → ReLU → [batch, 1024 dimensions] → Classifier → [batch, 2 dimensions]
# Each of the 12 BERT layers contains two sub-components: 1. Multi-Head Self-Attention and 2. Feed-Forward Neural Network (with 2 layers)

class BertClassifier(nn.Module):
    def __init__(self, num_classes):
        super(BertClassifier, self).__init__()
        self.bert = AutoModel.from_pretrained('bert-base-cased')
        self.dense = nn.Linear(768, 1024)       # 768 is BERT's hidden size
        self.relu = nn.ReLU()
        self.classifier = nn.Linear(1024, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output    # equivalent to bert.bert(...)[1]
        x = self.relu(self.dense(pooled_output))
        return self.classifier(x)                # no softmax — CrossEntropyLoss handles it

num_classes = int(labels.max()) + 1
model = BertClassifier(num_classes=num_classes)

In [ ]:
model

## Let's train the model!

Before passing data into its transformer layers, BERT adds up the following into one embedding:

*   A 768-dimensional vector converted from the input_id integers set by the tokenizer from BERT's predefined vocabulary.
*   A learned embedding for each position (0 to 511) so BERT knows the order of tokens (different from the fixed positional encoding applied in the original Attention is all you need paper)
* Segment embeddings — indicates which sentence a token belongs to

In [ ]:
# "Adam optimization is a stochastic gradient descent method that is based on adaptive estimation of first-order and second-order moments."
# learn more about the Adam Optimizer: https://keras.io/api/optimizers/adam/
# "The step size is determined by the learning rate. It determines how fast or slow the optimizer descends the error curve."
# https://towardsdatascience.com/how-to-choose-the-optimal-learning-rate-for-neural-networks-362111c5c783
# "a lower learning rate, such as 2e−5, is necessary to make BERT overcome the catastrophic forgetting problem" Sun et al (2019) https://doi.org/10.1007/978-3-030-32381-3_16

from torch.optim import Adam
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

optimizer = Adam(model.parameters(), lr=1e-5)
loss_fn = nn.CrossEntropyLoss()  # equivalent of CategoricalCrossentropy, works with class indices

# Training loop
epochs = 3

for epoch in range(epochs):
    # --- Training ---
    model.train()
    total_train_loss, total_train_acc = 0, 0

    for input_ids, masks, labels in train_loader:
        input_ids, masks, labels = input_ids.to(device), masks.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask=masks)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()
        preds = outputs.argmax(dim=1)
        total_train_acc += (preds == labels).float().mean().item()

    avg_train_loss = total_train_loss / len(train_loader)
    avg_train_acc = total_train_acc / len(train_loader)

    # --- Validation ---
    model.eval()
    total_val_loss, total_val_acc = 0, 0

    with torch.no_grad():  # no gradient computation needed for validation
        for input_ids, masks, labels in val_loader:
            input_ids, masks, labels = input_ids.to(device), masks.to(device), labels.to(device)

            outputs = model(input_ids, attention_mask=masks)
            loss = loss_fn(outputs, labels)

            total_val_loss += loss.item()
            preds = outputs.argmax(dim=1)
            total_val_acc += (preds == labels).float().mean().item()

    avg_val_loss = total_val_loss / len(val_loader)
    avg_val_acc = total_val_acc / len(val_loader)

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"  Train Loss: {avg_train_loss:.4f} | Train Acc: {avg_train_acc:.4f}")
    print(f"  Val Loss:   {avg_val_loss:.4f} | Val Acc:   {avg_val_acc:.4f}")

### Let's see results for previously unseen text

#### Function to take some text, split and encode using BERT tokenizer

In [ ]:
from transformers import BertTokenizer
import torch

tokenizer = BertTokenizer.from_pretrained('bert-base-cased')

def prep_data(text):
    tokens = tokenizer(text, max_length=512, truncation=True, padding='max_length',
                       add_special_tokens=True, return_token_type_ids=False,
                       return_tensors='pt')
    return {
        'input_ids': tokens['input_ids'].long(),
        'attention_mask': tokens['attention_mask'].long()
    }

#### Function to use the trained BERT model to predict the class of some new text

In [ ]:
def get_model_prediction(input_text):
    tokenized_text = prep_data(input_text)

    model.eval()
    with torch.no_grad():
        input_ids = tokenized_text['input_ids'].to(device)
        attention_mask = tokenized_text['attention_mask'].to(device)
        logits = model(input_ids, attention_mask=attention_mask)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]  # convert to probabilities

    top_class = list(class_label_dict.keys())[list(class_label_dict.values()).index(np.argmax(probs))]
    confidence_level = np.amax(probs)
    print(top_class)
    return top_class, confidence_level

These two examples are taken from a completely different debate:
https://www.theyworkforyou.com/debates/?id=2023-06-27b.170.2

In [ ]:
text_to_predict = 'I beg to move, That this House is extremely concerned that, under this Conservative Government, average \
                  mortgage costs will be increasing by £2,900 per year, with a typical household in the UK paying over £2,000 more per year \
                  than in France and over £1,000 more than in Ireland and Belgium, and that renters face huge increases in rent payments;.'
get_model_prediction(text_to_predict)

In [ ]:
text_to_predict = 'I am here to account for what has happened in the UK. Obviously, there are differences—[Interruption.] If I may answer. \
                  There are differences across the EU and the US. What I am telling the House, which is quite transparently clear, \
                  is that inflationary pressures are affecting all economies at the moment, and it is my responsibility to account for what we are doing as a Government. \
                  holocaust memorial and education centre. I understand that the Standing Orders Committee has considered \
                  the progress of the Holocaust Memorial Bill, which will bring both the much-needed and expected education centre \
                  and the memorial to fruition. Can my right hon. Friend provide a progress report on that Bill, but also on the \
                  long-promised boycotts, divestment and sanctions Bill that the Government have promised to bring forward?'
get_model_prediction(text_to_predict)

Exercise 5: Run the model on the text below, is it correct?

In [ ]:
new_text_to_predict = 'May I return briefly to the point made by my hon. Friend Barbara Keeley? Last time I \
                        asked the Economic Secretary to the Treasury about the number of renters estimated to be \
                        impacted by this situation, he did not have an answer. Do Ministers on the Treasury front bench \
                        have an answer today on how many renters will be affected by this crisis?'

## Let's look at our held out test data from the same debates

In [ ]:
test_df # remember we split out this test set of labelled data earlier

## Build confusion matrix

In [ ]:
# Prepare a list of the y_true items (the real labels)
y_true = test_df['Party'].tolist()
y_true

In [ ]:
len(y_true)

In [ ]:
for some_text in test_df['Text']:
  print(some_text)

In [ ]:
#check text length and truncate if it is over 512 length
test_long_text = 0
test_id = 0
test_truncated_text = 0
test_all_text = [] # we will store of the text from the dataframe in this list
for some_text in test_df.Text:
  test_id+=1
  if len(some_text)>512:
    #print(some_text)
    test_long_text+=1
    test_truncated_text = some_text[:512]
    test_all_text.append(test_truncated_text)
  else:
    test_all_text.append(some_text)
print("There are a total of", test_long_text, "text units that are too long, these were chopped!")

In [ ]:
test_all_text

In [ ]:
test_df['Text'] = test_all_text # replace the text in the test dataframe with the truncated version

In [ ]:
test_df

In [ ]:
y_pred = []
y_pred_confidence=[]
for some_text in test_df['Text']:
  result = get_model_prediction(some_text)
  print(result)
  print(type(result))
  y_pred.append(result[0])
  y_pred_confidence.append(result[1])

In [ ]:
y_pred #the predictions from the model

In [ ]:
len(y_pred)

### Let's see how well our model did on the test set

In [ ]:
from sklearn import metrics
print(metrics.classification_report(y_true,y_pred))

Excercise 5: What is precision? Which party has better precision in our model?

Exercise 6: What is recall? Which party has better recall?

### Let's create a confusion matrix to see how results are broken down

In [ ]:
mylabels = list(set(y_true))
mylabels

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_true, y_pred, labels=mylabels)
cm

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
cm = confusion_matrix(y_true, y_pred, labels=mylabels)

cmd = ConfusionMatrixDisplay(cm, display_labels=mylabels)
fig, ax = plt.subplots(figsize=(10,10))
cmd.plot(ax=ax)

Exercise 7: how many Labour party debate text items were incorrectly predicted as Conservative?

Exercise 8: how many Conservative party debate items were correctly predicted?

### Let's manually examine the results: where is the model going wrong?

In [ ]:
test_df['y_pred'] = y_pred # add the predictions to the dataframe as a new column
test_df['confidence'] = y_pred_confidence # add the model confidence for each prediction as well
test_df['accuracy'] = np.where((test_df['Party']== test_df['y_pred']), "Correct", "Incorrect") # Specifically label the accuracy of each prediction in the dataframe

test_df

In [ ]:
wrong_results = test_df.loc[test_df['accuracy'] == "Incorrect"]
len(wrong_results)

In [ ]:
wrong_results

In [ ]:
test_labels_np = np.array(y_true)
all_preds_np   = np.array(y_pred)
all_probs_np   = np.array(y_pred_confidence)

In [ ]:
all_probs_np

In [ ]:
class_label_dict

In [ ]:

# Use class_label_dict as the single source of truth
test_labels_encoded = np.array([class_label_dict[label] for label in test_labels_np])


In [ ]:
test_labels_encoded

In [ ]:
# ================================================
# Confidence Distribution by Class
# ================================================
plt.figure(figsize=(8, 4))
for label, name, color in [(0, 'Conservative', 'steelblue'), (1, 'Labour', 'coral')]:
    mask = test_labels_encoded == label
    plt.hist(all_probs_np[mask], bins=20, alpha=0.6, label=name, color=color)
plt.axvline(0.5, color='black', linestyle='--', label='Decision boundary')
plt.xlabel('Predicted Probability')
plt.ylabel('Count')
plt.title('Confidence Distribution by True Class')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
incorrect_mask = all_preds_np != test_labels_np
incorrect_mask

In [ ]:
# ================================================
# Confidence of Correct vs Incorrect Predictions
# ================================================
correct_mask   = ~incorrect_mask
correct_confs  = all_probs_np[correct_mask]
incorrect_confs = all_probs_np[incorrect_mask]

# Remap to distance from decision boundary (0.5)
correct_margin   = np.abs(correct_confs - 0.5)
incorrect_margin = np.abs(incorrect_confs - 0.5)

plt.figure(figsize=(6, 4))
plt.boxplot([correct_margin, incorrect_margin],
            tick_labels =['Correct', 'Incorrect'],
            patch_artist=True,
            boxprops=dict(facecolor='steelblue', alpha=0.6))
plt.ylabel('Confidence margin from 0.5')
plt.title('Prediction Confidence: Correct vs Incorrect')
plt.tight_layout()
plt.show()

print(f"\nMean confidence margin — Correct:   {correct_margin.mean():.3f}")
print(f"Mean confidence margin — Incorrect: {incorrect_margin.mean():.3f}")

In [ ]:
# ================================================
# Confidence by Class: Correct vs Incorrect
# ================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, (label, name, color) in zip(axes, [(0, 'Conservative', 'steelblue'), (1, 'Labour', 'coral')]):
    class_mask = test_labels_encoded == label

    correct_confs   = np.abs(all_probs_np[class_mask & ~incorrect_mask] - 0.5)
    incorrect_confs = np.abs(all_probs_np[class_mask &  incorrect_mask] - 0.5)

    bp = ax.boxplot([correct_confs, incorrect_confs],
                    tick_labels =['Correct', 'Incorrect'],
                    patch_artist=True,
                    boxprops=dict(alpha=0.7))

    # Colour boxes individually
    bp['boxes'][0].set_facecolor(color)
    bp['boxes'][1].set_facecolor('lightgrey')

    ax.set_title(f'{name}')
    ax.set_ylabel('Confidence margin from 0.5')
    ax.set_xlabel('Prediction outcome')

    print(f"{name} — Correct:   {correct_confs.mean():.3f} | Incorrect: {incorrect_confs.mean():.3f}")

fig.suptitle('Prediction Confidence by Class: Correct vs Incorrect', fontsize=13)
plt.tight_layout()
plt.show()

Exercise 9: How can we interpret these results?

## Let's save our model to use again another time

In [ ]:
# Mount your google drive

from google.colab import drive
import os

# ✅ Mount Google Drive
drive.mount('/content/drive')


In [ ]:
# ✅ Specify save folder — change this to your preferred path in google drive, click on the three dots to copy the path from the file window
save_dir = '/content/drive/MyDrive/.....'

os.makedirs(save_dir, exist_ok=True)

# ✅ Save model weights and full training info
torch.save({
    'model_state_dict':     model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'num_classes':          num_classes,
    'class_label_dict':     class_label_dict,  # single source of truth
}, os.path.join(save_dir, 'party_model.pt'))


print(f"Model saved to: {save_dir}/party_model.pt")

In [ ]:
# Now go and check your folder in Google Drive